In [1]:
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, accuracy_score
import joblib
import os

In [ ]:
url_alay = "https://raw.githubusercontent.com/nasalsabila/kamus-alay/master/colloquial-indonesian-lexicon.csv"
url_pos = "https://raw.githubusercontent.com/fajri91/InSet/master/positive.tsv"
url_neg = "https://raw.githubusercontent.com/fajri91/InSet/master/negative.tsv"

print("Downloading Kamus Alay...")
df_alay = pd.read_csv(url_alay)
kamus_alay = dict(zip(df_alay['slang'], df_alay['formal']))

print("Downloading InSet Lexicon...")
def load_inset(url):
    df = pd.read_csv(url, sep='\t')
    return dict(zip(df['word'], df['weight']))

lexicon_pos = load_inset(url_pos)
lexicon_neg = load_inset(url_neg)

print("Loading playstore reviews...")
df = pd.read_csv("playstore_reviews.csv")
df = df.dropna(subset=["content"]).reset_index(drop=True)


Loading playstore reviews...


In [3]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    words = text.split()
    words = [kamus_alay.get(w, w) for w in words]
    return ' '.join(words)

def label_sentiment(text):
    words = text.split()
    total_score = 0
    for w in words:
        total_score += lexicon_pos.get(w, 0)
        total_score += lexicon_neg.get(w, 0)
    
    if total_score > 0:
        return 'positive'
    elif total_score < 0:
        return 'negative'
    else:
        return 'neutral'

print("Cleaning text...")
df['clean_content'] = df['content'].apply(clean_text)
df = df[df['clean_content'].str.strip().str.len() > 3]

print("Labeling with lexicon...")
df['label'] = df['clean_content'].apply(label_sentiment)


Cleaning text...
Labeling with lexicon...


In [ ]:
print("Extracting features and training model...")
vectorizer_word = TfidfVectorizer(
    lowercase=True,
    sublinear_tf=True,
    ngram_range=(1, 2),
    min_df=3,
    max_df=0.95,
)
vectorizer_char = TfidfVectorizer(
    analyzer="char",
    sublinear_tf=True,
    ngram_range=(3, 5),
    min_df=3,
    max_df=1.0,
)
features = FeatureUnion([
    ("word", vectorizer_word),
    ("char", vectorizer_char),
])
model = LinearSVC(C=1.0, class_weight='balanced')
pipeline = Pipeline([
    ("features", features),
    ("clf", model),
])

X = df["clean_content"].astype(str)
y = df["label"].astype(str)

class_counts = y.value_counts()
valid_classes = class_counts[class_counts > 5].index
mask = y.isin(valid_classes)
X = X[mask]
y = y[mask]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"Accuracy: {acc}")
print(classification_report(y_test, y_pred, digits=4))


Extracting features and training model...


Accuracy: 0.8799414348462665
              precision    recall  f1-score   support

    negative     0.9087    0.9497    0.9287       398
     neutral     0.7273    0.6588    0.6914        85
    positive     0.8789    0.8350    0.8564       200

    accuracy                         0.8799       683
   macro avg     0.8383    0.8145    0.8255       683
weighted avg     0.8774    0.8799    0.8780       683



In [5]:
os.makedirs("models", exist_ok=True)
joblib.dump(pipeline, "models/sentiment_playstore.joblib")
with open("models/metrics.txt", "w", encoding="utf-8") as f:
    f.write(f"accuracy={acc}\n")


In [6]:
def predict_sentiment(text):
    clean = clean_text(text)
    return pipeline.predict([clean])[0]

print(predict_sentiment("Aplikasi ini sangat bagus dan mudah digunakan"))
print(predict_sentiment("Sering error dan lemot"))
print(predict_sentiment("Biasa saja, tidak terlalu istimewa"))


negative
negative
negative
